# 02 — Backtest

Full vectorbt backtest of the BTC Production Cost Strategy.

- Entry: PCR < 0.9 **and** RSI(14) < 45
- Exit:  PCR > 2.0 **and** RSI(14) > 55
- Comparison against Buy-and-Hold benchmark

In [ ]:
import sys
sys.path.insert(0, '..')

import vectorbt as vbt
import pandas as pd
import matplotlib.pyplot as plt

from data.fetch_price import fetch_price_data
from data.fetch_onchain import fetch_hashrate
from models.production_cost import compute_production_cost
from signals.signal_engine import compute_signals

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Prepare data and signals

In [ ]:
price_df  = fetch_price_data()
hr_df     = fetch_hashrate()
cost_df   = compute_production_cost(hr_df)
sigs      = compute_signals(price_df, cost_df)

close   = sigs['close']
entries = sigs['entry_signal'].fillna(False)
exits   = sigs['exit_signal'].fillna(False)

print(f"Entries: {entries.sum()}  Exits: {exits.sum()}  Days: {len(close)}")

## 2. Run vectorbt simulation

In [ ]:
pf = vbt.Portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    init_cash=10_000,
    fees=0.001,
    slippage=0.001,
    freq='1D',
)
pf.stats()

## 3. Equity Curve vs Buy-and-Hold

In [ ]:
bh_value = (close / close.iloc[0]) * 10_000

fig, ax = plt.subplots()
pf.value().rename('Strategy').plot(ax=ax)
bh_value.rename('Buy & Hold').plot(ax=ax, linestyle='--')
ax.set_title('Strategy vs Buy-and-Hold')
ax.set_ylabel('Portfolio Value (USD)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Drawdown

In [ ]:
fig, ax = plt.subplots()
pf.drawdown().rename('Strategy Drawdown').plot(ax=ax, color='red')
ax.set_title('Strategy Drawdown')
ax.set_ylabel('Drawdown (%)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))
plt.tight_layout()
plt.show()

## 5. Trade list

In [ ]:
pf.trades.records_readable.head(20)